In [4]:
import pandas as pd
import os

#file paths
data_dir = os.path.join("..", "data")
perugia_path = os.path.join(data_dir, "POSAS_2026_it_054_Perugia.csv")
terni_path = os.path.join(data_dir, "POSAS_2026_it_055_Terni.csv")

income_path = os.path.join(data_dir, "Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv")

# Load and merge the ISTAT provincial data
perugia_df = pd.read_csv(perugia_path, sep=';', skiprows=1)
terni_df = pd.read_csv(terni_path, sep=';', skiprows=1)
umbria_demographics = pd.concat([perugia_df, terni_df], ignore_index=True)

# Load the MEF income data
umbria_income = pd.read_csv(income_path, sep=';')

# Verify the merge
print(umbria_demographics.columns.tolist())
display(umbria_demographics.head(3))

['Codice comune', 'Comune', 'Età', 'Totale maschi', 'Totale femmine', 'Totale']


,Codice comune,Comune,Età,Totale maschi,Totale femmine,Totale
0,054001,Assisi,0.0,77.0,55.0,132.0
1,054001,Assisi,1.0,81.0,77.0,158.0
2,054001,Assisi,2.0,81.0,86.0,167.0


In [5]:
# Standardize names of comuni
umbria_demographics['Comune_Key'] = umbria_demographics['Comune'].astype(str).str.strip().str.upper()
umbria_income['Comune_Key'] = umbria_income['Denominazione Comune'].astype(str).str.strip().str.upper()

# BULLETPROOF FILTER: Convert 'Età' to numeric (text like 'TOTALE' becomes NaN)
umbria_demographics['Età_Num'] = pd.to_numeric(umbria_demographics['Età'], errors='coerce')

# Keep only rows where the age is a realistic human age (drops 999 and NaNs)
umbria_filtered = umbria_demographics[umbria_demographics['Età_Num'] < 150]

# Group ages together using the FILTERED dataset
istat_aggreg = umbria_filtered.groupby('Comune_Key', as_index=False)['Totale'].sum()
istat_aggreg = istat_aggreg.rename(columns={'Totale': 'Popolazione Totale'})

# Merge based on the comune key
dataset_completo = pd.merge(istat_aggreg, umbria_income, on='Comune_Key', how='inner')

# Verify merge and check Assisi's population
print(f"Righe totali dopo il merge: {len(dataset_completo)}")
selected_row = dataset_completo[dataset_completo['Comune_Key'] == 'ASSISI']
display(selected_row[['Comune_Key', 'Popolazione Totale', 'Numero contribuenti']])

display(dataset_completo)

Righe totali dopo il merge: 90


,Comune_Key,Popolazione Totale,Numero contribuenti
5,ASSISI,27400.0,20543


,Comune_Key,Popolazione Totale,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti,...,Reddito complessivo da 15000 a 26000 euro - Ammontare in euro,Reddito complessivo da 26000 a 55000 euro - Frequenza,Reddito complessivo da 26000 a 55000 euro - Ammontare in euro,Reddito complessivo da 55000 a 75000 euro - Frequenza,Reddito complessivo da 55000 a 75000 euro - Ammontare in euro,Reddito complessivo da 75000 a 120000 euro - Frequenza,Reddito complessivo da 75000 a 120000 euro - Ammontare in euro,Reddito complessivo oltre 120000 euro - Frequenza,Reddito complessivo oltre 120000 euro - Ammontare in euro,
0,ACQUASPARTA,4330.0,2024,A045,55001,ACQUASPARTA,TR,Umbria,10,3185,...,19088255.0,996.0,33813067.0,49.0,3191237.0,43.0,3933983.0,12.0,1740693.0,NaN
1,ALLERONA,1679.0,2024,A207,55002,ALLERONA,TR,Umbria,10,1298,...,8839206.0,361.0,12155914.0,19.0,1224372.0,9.0,838590.0,4.0,678547.0,NaN
2,ALVIANO,1318.0,2024,A242,55003,ALVIANO,TR,Umbria,10,989,...,5510733.0,320.0,11129177.0,9.0,582784.0,NaN,NaN,NaN,NaN,NaN
3,AMELIA,11404.0,2024,A262,55004,AMELIA,TR,Umbria,10,8475,...,48610009.0,2655.0,90544404.0,187.0,11902647.0,148.0,13655205.0,NaN,NaN,NaN
4,ARRONE,2504.0,2024,A439,55005,ARRONE,TR,Umbria,10,1824,...,11086017.0,572.0,19320840.0,26.0,1653107.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,TUORO SUL TRASIMENO,3793.0,2024,L466,54055,TUORO SUL TRASIMENO,PG,Umbria,10,2937,...,19244778.0,769.0,25988331.0,43.0,2715885.0,33.0,3021846.0,11.0,2337469.0,NaN
86,UMBERTIDE,16207.0,2024,D786,54056,UMBERTIDE,PG,Umbria,10,12098,...,86500170.0,3686.0,122303421.0,193.0,12252775.0,150.0,13982320.0,87.0,16816394.0,NaN
87,VALFABBRICA,3225.0,2024,L573,54057,VALFABBRICA,PG,Umbria,10,2447,...,17982863.0,625.0,20813069.0,40.0,2549354.0,20.0,1786626.0,9.0,1257325.0,NaN
88,VALLO DI NERA,317.0,2024,L627,54058,VALLO DI NERA,PG,Umbria,10,245,...,1406449.0,75.0,2409593.0,NaN,NaN,NaN,NaN,0.0,0.0,NaN


**Feature Distinction: Total Population vs. Taxpayers**
When comparing the demographic and economic features, there is a consistent gap between `Popolazione Totale` and `Numero contribuenti`. The total population counts every registered resident in the municipality. The taxpayer metric only accounts for individuals who filed an income tax return (IRPEF). This structural gap filters out children, students, and adults without independent declarable income, making the taxpayer count a useful proxy for the active workforce and income-generating base of the town.

In [10]:
#From OMI Data, prices in €/m² for each macro-area (from OMI tables 11 and 15)
omi_prices = {
    # Perugia Macro-areas (Table 11)
    'ALTO TEVERE': 782,
    'ASSISIATE': 982,
    'COLLI DEL TRASIMENO': 802,
    'EUGUBINO-GUALDESE': 705,
    'FOLIGNATE': 886,
    'MEDIA VALLE DEL TEVERE': 825,
    'MONTI MARTANI': 556,
    'PERUGINO': 940,
    'SPOLETINO': 958,
    'PERUGIA': 1075, # Solo capoluogo
    
    # Terni Macro-areas (Table 15)
    'AMERINO': 782,
    'NARNESE': 832,
    'ORVIETANO': 898,
    'VALNERINA': 743,
    'TERNI': 895 # Solo capoluogo
}

comune_to_macroarea = {
    # --- PROVINCIA DI PERUGIA ---
    # Alto Tevere
    'CITERNA': 'ALTO TEVERE', 'CITTA\' DI CASTELLO': 'ALTO TEVERE', 'LISCIANO NICCONE': 'ALTO TEVERE', 
    'MONTE SANTA MARIA TIBERINA': 'ALTO TEVERE', 'MONTONE': 'ALTO TEVERE', 'PIETRALUNGA': 'ALTO TEVERE', 
    'SAN GIUSTINO': 'ALTO TEVERE', 'UMBERTIDE': 'ALTO TEVERE',
    
    # Assisiate
    'ASSISI': 'ASSISIATE', 'BASTIA UMBRA': 'ASSISIATE', 'BETTONA': 'ASSISIATE', 'CANNARA': 'ASSISIATE',
    
    # Colli del Trasimeno
    'CASTIGLIONE DEL LAGO': 'COLLI DEL TRASIMENO', 'CITTA\' DELLA PIEVE': 'COLLI DEL TRASIMENO', 
    'MAGIONE': 'COLLI DEL TRASIMENO', 'PACIANO': 'COLLI DEL TRASIMENO', 'PANICALE': 'COLLI DEL TRASIMENO', 
    'PASSIGNANO SUL TRASIMENO': 'COLLI DEL TRASIMENO', 'PIEGARO': 'COLLI DEL TRASIMENO', 
    'TUORO SUL TRASIMENO': 'COLLI DEL TRASIMENO',
    
    # Eugubino-Gualdese
    'COSTACCIARO': 'EUGUBINO-GUALDESE', 'FOSSATO DI VICO': 'EUGUBINO-GUALDESE', 'GUALDO TADINO': 'EUGUBINO-GUALDESE', 
    'GUBBIO': 'EUGUBINO-GUALDESE', 'SCHEGGIA E PASCELUPO': 'EUGUBINO-GUALDESE', 'SIGILLO': 'EUGUBINO-GUALDESE', 
    'VALFABBRICA': 'EUGUBINO-GUALDESE',
    
    # Folignate
    'BEVAGNA': 'FOLIGNATE', 'FOLIGNO': 'FOLIGNATE', 'MONTEFALCO': 'FOLIGNATE', 'NOCERA UMBRA': 'FOLIGNATE', 
    'SPELLO': 'FOLIGNATE', 'TREVI': 'FOLIGNATE', 'VALTOPINA': 'FOLIGNATE',
    
    # Media Valle del Tevere
    'COLLAZZONE': 'MEDIA VALLE DEL TEVERE', 'DERUTA': 'MEDIA VALLE DEL TEVERE', 'FRATTA TODINA': 'MEDIA VALLE DEL TEVERE', 
    'MARSCIANO': 'MEDIA VALLE DEL TEVERE', 'MONTE CASTELLO DI VIBIO': 'MEDIA VALLE DEL TEVERE', 'TODI': 'MEDIA VALLE DEL TEVERE',
    
    # Monti Martani
    'CASTEL RITALDI': 'MONTI MARTANI', 'GIANO DELL\'UMBRIA': 'MONTI MARTANI', 'MASSA MARTANA': 'MONTI MARTANI',
    
    # Perugino
    'CORCIANO': 'PERUGINO', 'PERUGIA': 'PERUGIA', 'TORGIANO': 'PERUGINO',
    
    # Spoletino
    'CAMPELLO SUL CLITUNNO': 'SPOLETINO', 'GUALDO CATTANEO': 'SPOLETINO', 'SPOLETO': 'SPOLETINO',
    
    # Val Nerina (Perugia)
    'CASCIA': 'VAL NERINA', 'CERRETO DI SPOLETO': 'VAL NERINA', 'MONTELEONE DI SPOLETO': 'VAL NERINA', 
    'NORCIA': 'VAL NERINA', 'POGGIODOMO': 'VAL NERINA', 'PRECI': 'VAL NERINA', 'SANT\'ANATOLIA DI NARCO': 'VAL NERINA', 
    'SCHEGGINO': 'VAL NERINA', 'SELLANO': 'VAL NERINA', 'VALLO DI NERA': 'VAL NERINA',

    # --- PROVINCIA DI TERNI ---
    # Amerino
    'ACQUASPARTA': 'AMERINO', 'ALVIANO': 'AMERINO', 'AMELIA': 'AMERINO', 'ATTIGLIANO': 'AMERINO', 
    'AVIGLIANO UMBRO': 'AMERINO', 'GIOVE': 'AMERINO', 'GUARDEA': 'AMERINO', 'LUGNANO IN TEVERINA': 'AMERINO', 
    'MONTECASTRILLI': 'AMERINO', 'MONTECCHIO': 'AMERINO', 'PENNA IN TEVERINA': 'AMERINO',
    
    # Narnese
    'CALVI DELL\'UMBRIA': 'NARNESE', 'NARNI': 'NARNESE', 'OTRICOLI': 'NARNESE', 
    'SAN GEMINI': 'NARNESE', 'STRONCONE': 'NARNESE',
    
    # Orvietano
    'ALLERONA': 'ORVIETANO', 'BASCHI': 'ORVIETANO', 'CASTEL GIORGIO': 'ORVIETANO', 'CASTEL VISCARDO': 'ORVIETANO', 
    'FABRO': 'ORVIETANO', 'FICULLE': 'ORVIETANO', 'MONTEGABBIONE': 'ORVIETANO', 'MONTELEONE D\'ORVIETO': 'ORVIETANO', 
    'ORVIETO': 'ORVIETANO', 'PARRANO': 'ORVIETANO', 'PORANO': 'ORVIETANO', 'SAN VENANZO': 'ORVIETANO',
    
    # Terni Capoluogo
    'TERNI': 'TERNI',
    
    # Valnerina (Terni)
    'ARRONE': 'VALNERINA', 'FERENTILLO': 'VALNERINA', 'MONTEFRANCO': 'VALNERINA', 'POLINO': 'VALNERINA'
}

dataset_completo['Macroarea_OMI'] = dataset_completo['Comune_Key'].map(comune_to_macroarea)

dataset_completo['Prezzo_Medio_Mq'] = dataset_completo['Macroarea_OMI'].map(omi_prices)

display(dataset_completo[['Comune_Key', 'Macroarea_OMI', 'Prezzo_Medio_Mq']].head(10))


,Comune_Key,Macroarea_OMI,Prezzo_Medio_Mq
0,ACQUASPARTA,AMERINO,782.0
1,ALLERONA,ORVIETANO,898.0
2,ALVIANO,AMERINO,782.0
3,AMELIA,AMERINO,782.0
4,ARRONE,VALNERINA,743.0
5,ASSISI,ASSISIATE,982.0
6,ATTIGLIANO,AMERINO,782.0
7,AVIGLIANO UMBRO,AMERINO,782.0
8,BASCHI,ORVIETANO,898.0
9,BASTIA UMBRA,ASSISIATE,982.0


In [ ]:
from sklearn.impute import SimpleImputer

# 1. Identify the pre-calculated MEF Total Income column
income_col = [c for c in dataset_completo.columns if 'Reddito complessivo' in c and 'Ammontare' in c]

if not income_col:
    print("Colonne di ammontare disponibili nel dataset MEF:")
    print([c for c in dataset_completo.columns if 'Ammontare' in c])
else:
    col_reddito = income_col[0]
    print(f"Colonna utilizzata per il reddito totale: {col_reddito}")
    
    # 2. Convert to numeric (forces any empty cells or text into NaN)
    dataset_completo[col_reddito] = pd.to_numeric(dataset_completo[col_reddito], errors='coerce')
    dataset_completo['Popolazione Totale'] = pd.to_numeric(dataset_completo['Popolazione Totale'], errors='coerce')
    dataset_completo['Numero contribuenti'] = pd.to_numeric(dataset_completo['Numero contribuenti'], errors='coerce')
    
    # 3. Impute NaNs using the median strategy
    imputer = SimpleImputer(strategy="median")
    
    # Apply the imputer to all three core numerical columns to ensure no NaNs remain
    cols_to_impute = [col_reddito, 'Popolazione Totale', 'Numero contribuenti']
    dataset_completo[cols_to_impute] = imputer.fit_transform(dataset_completo[cols_to_impute])
    
    # 4. Calculate Pro Capita metrics safely
    dataset_completo['Reddito_Pro_Capite'] = (
        dataset_completo[col_reddito] / dataset_completo['Popolazione Totale']
    )
    
    dataset_completo['Reddito_Medio_Contribuente'] = (
        dataset_completo[col_reddito] / dataset_completo['Numero contribuenti']
    )
    
    # 5. Display the cleaned results
    cols_view = ['Comune_Key', 'Popolazione Totale', 'Numero contribuenti', 
                 'Reddito_Pro_Capite', 'Reddito_Medio_Contribuente', 'Prezzo_Medio_Mq']
    display(dataset_completo[cols_view].head(10))

Colonna utilizzata per il reddito totale: Reddito complessivo - Ammontare in euro


,Comune_Key,Popolazione Totale,Numero contribuenti,Reddito_Pro_Capite,Reddito_Medio_Contribuente,Prezzo_Medio_Mq
0,ACQUASPARTA,4330.0,3185.0,16046.300693,21814.908006,782.0
1,ALLERONA,1679.0,1298.0,16246.483026,21015.288906,898.0
2,ALVIANO,1318.0,989.0,15837.377845,21105.828109,782.0
3,AMELIA,11404.0,8475.0,17233.221764,23189.104543,782.0
4,ARRONE,2504.0,1824.0,15568.603834,21372.688596,743.0
5,ASSISI,27400.0,20543.0,18179.912482,24248.143017,982.0
6,ATTIGLIANO,1962.0,1359.0,14546.903670,21001.490066,782.0
7,AVIGLIANO UMBRO,2308.0,1695.0,16454.762998,22405.659587,782.0
8,BASCHI,2544.0,1930.0,16488.499607,21734.063731,898.0
9,BASTIA UMBRA,21497.0,16113.0,17861.055961,23829.151617,982.0
